# Tercer examen  
Probabilidad y Estadística para la Inteligencia Artificial  

**Docente**: Camilo Argoty  
**Alumno**: Elian Pinzas  


_Carrera de Especialización en Inteligencia Artificial_  
_Laboratorio de Sistemas Embebidos_  
_Facultad de Ingeniería_  
_Universidad de Buenos Aires_  



Siguiendo con la historia de Don Francisco, con el tiempo y gracias a los análisis de Matías, el pequeño comerciante de barrio cuenta hoy con 5 supermercados: 'Santa Ana', 'La Floresta', 'Los Cedros', 'Palermo' y 'Córdoba'.

También Matías ha avanzado en la Especialización en Inteligencia Artificial. Un día Don Francisco le plantea algunas inquietudes adicionales:

Don Francisco quiere entender mejor la afluencia de clientes por mes del supermercado 'Santa Ana'. Más aún, Don Francisco no sabe si puede estar seguro de que la afluencia de clientes son las mismas en todos los supermercados o si hay alguno que se comporte mejor que los demás, y si alguna de las tiendas necesita más atención porque está recibiendo menos clientes que las de las otras.

1- Crear una simulación del número de clientes diarios que van a los almacenes de Don Francisco, usando distribuciones Poisson, entre los años 2023, 2024 y 2025. En cada fecha, el parámetro $\lambda_{t}$ debe ser la suma de los siguientes efectos:


Efecto anual:
| Año | Efecto |
|-|-|
| 2023 | 1000 |
| 2024 | 1500 |
| 2025 | 2000 |

Efecto mensual:
| Mes | Efecto |
|-|-|
| Enero | 1000 |
| Febrero | 1500 |
| Marzo | 2000 |
| Abril | 2000 |
| Mayo | 2500 |
| Junio | 2500 |
| Julio | 3000 |
| Agosto | 2500 |
| Septiembre | 2500 |
| Octubre | 2000 |
| Noviembre | 1500 |
| Diciembre | 1000 |

Efecto diario:
| Día | Efecto |
|-|-|
| Domingo | 1000 |
| Lunes | 2000 |
| Martes | 3000 |
| Miércoles | 3500 |
| Jueves | 3000 |
| Viernes | 2000 |
| Sábado | 1000 |


Efecto por tienda:
| Tienda | Efecto |
|-|-|
| Santa Ana | 5000 |
| La Floresta | 2000 |
| Los Cedros | 3000 |
| Palermo | 1000 |
| Córdoba | 3000 |



In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [2]:
# 1. Definición del rango de fechas (2023 - 2025)
fechas = pd.date_range(start='2023-01-01', end='2025-12-31', freq='D')

# 2. Diccionarios con los efectos individuales
efecto_anio = {
    2023: 1000, 
    2024: 1500, 
    2025: 2000
}

efecto_mes = {
    1: 1000,  # Enero
    2: 1500,  # Febrero
    3: 2000,  # Marzo
    4: 2000,  # Abril
    5: 2500,  # Mayo
    6: 2500,  # Junio
    7: 3000,  # Julio
    8: 2500,  # Agosto
    9: 2500,  # Septiembre
    10: 2000, # Octubre
    11: 1500, # Noviembre
    12: 1000  # Diciembre
}

efecto_dia = {
    'Sunday': 1000,
    'Monday': 2000,
    'Tuesday': 3000,
    'Wednesday': 3500,
    'Thursday': 3000,
    'Friday': 2000,
    'Saturday': 1000
}

efecto_tienda = {
    'Santa Ana': 5000,
    'La Floresta': 2000,
    'Los Cedros': 3000,
    'Palermo': 1000,
    'Córdoba': 3000
}

# 3. Generación de la simulación
registros = []
np.random.seed(42)

for fecha in fechas:
    e_anio = efecto_anio[fecha.year]
    e_mes = efecto_mes[fecha.month]
    e_dia = efecto_dia[fecha.day_name()]
    
    for tienda, e_tienda in efecto_tienda.items():
        # Suma de efectos para obtener lambda_t[cite: 1]
        lambda_t = e_anio + e_mes + e_dia + e_tienda
        
        # Simulación según distribución Poisson[cite: 1]
        clientes = np.random.poisson(lam=lambda_t)
        
        registros.append({
            'Fecha': fecha,
            'Año': fecha.year,
            'Mes': fecha.month,
            'Día_Semana': fecha.day_name(),
            'Tienda': tienda,
            'Lambda_t': lambda_t,
            'Clientes': clientes
        })

# 4. Creación del DataFrame final
df_simulacion = pd.DataFrame(registros)

# Mostrar primeros registros
print(df_simulacion.head())

       Fecha   Año  Mes Día_Semana       Tienda  Lambda_t  Clientes
0 2023-01-01  2023    1     Sunday    Santa Ana      8000      7968
1 2023-01-01  2023    1     Sunday  La Floresta      5000      5049
2 2023-01-01  2023    1     Sunday   Los Cedros      6000      5911
3 2023-01-01  2023    1     Sunday      Palermo      4000      4018
4 2023-01-01  2023    1     Sunday      Córdoba      6000      6085


2- Con base en los datos generados, determinen intervalos de confianza empíricos para el supermercado 'Santa Ana' en cada mes, para significancias del 95\% y el 99\%.


In [3]:
# 1. Filtrar los datos para la tienda 'Santa Ana'[cite: 1]
df_santa_ana = df_simulacion[df_simulacion['Tienda'] == 'Santa Ana'].copy()

# Map opcional para mostrar los nombres de los meses en lugar de números
nombres_meses = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril',
    5: 'Mayo', 6: 'Junio', 7: 'Julio', 8: 'Agosto',
    9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
}

# 2. Función para calcular los intervalos de confianza empíricos
def calcular_ic(serie):
    # 95% de confianza (alpha = 5%)
    ic_95_inferior = np.percentile(serie, 2.5)
    ic_95_superior = np.percentile(serie, 97.5)
    
    # 99% de confianza (alpha = 1%)
    ic_99_inferior = np.percentile(serie, 0.5)
    ic_99_superior = np.percentile(serie, 99.5)
    
    return pd.Series({
        'Promedio': np.mean(serie),
        'IC_95_Inf': ic_95_inferior,
        'IC_95_Sup': ic_95_superior,
        'IC_99_Inf': ic_99_inferior,
        'IC_99_Sup': ic_99_superior
    })

# 3. Agrupar por mes y calcular los intervalos[cite: 1]
ic_santa_ana = df_santa_ana.groupby('Mes')['Clientes'].apply(calcular_ic).unstack()

# Formatear el índice con el nombre del mes
ic_santa_ana.index = ic_santa_ana.index.map(nombres_meses)

print(ic_santa_ana.round(2))

            Promedio  IC_95_Inf  IC_95_Sup  IC_99_Inf  IC_99_Sup
Mes                                                             
Enero        9748.02    7981.20   11500.00    7872.56   11680.20
Febrero     10207.42    8392.90   11846.30    8313.70   12094.40
Marzo       10669.53    8935.60   12488.70    8890.82   12559.66
Abril       10715.71    8891.88   12499.50    8834.69   12593.80
Mayo        11225.94    9410.00   12893.80    9379.26   12949.98
Junio       11197.66    9567.67   12953.40    9483.60   13116.15
Julio       11751.63    9910.20   13550.30    9860.18   13593.40
Agosto      11208.99    9467.50   13023.50    9383.28   13245.50
Septiembre  11206.38    9514.35   12982.65    9452.46   13131.09
Octubre     10751.47    8966.50   12451.10    8810.50   12533.54
Noviembre   10191.38    8445.08   11847.85    8350.17   11929.54
Diciembre    9700.14    7927.50   11535.30    7829.36   11602.60


3- De igual manera, realicen pruebas ANOVA para determinar si los clientes esperados de todas las tiendas son iguales o no, con significancia del 95\%.

In [4]:
# 1. Agrupar las observaciones de clientes por cada tienda
grupos_tiendas = [grupo['Clientes'].values for _, grupo in df_simulacion.groupby('Tienda')]

# 2. Ejecutar la prueba ANOVA de un factor
f_stat, p_value = stats.f_oneway(*grupos_tiendas)

print("=== PRUEBA ANOVA DE UN FACTOR ===")
print(f"Estadístico F: {f_stat:.4f}")
print(f"p-valor: {p_value:.4e}")

alpha = 0.05
if p_value < alpha:
    print("Conclusión: Se rechaza H0. Existen diferencias estadísticamente significativas "
          "en el número medio de clientes entre las tiendas.")
else:
    print("Conclusión: No se rechaza H0. No hay evidencia de diferencias significativas entre las tiendas.")

=== PRUEBA ANOVA DE UN FACTOR ===
Estadístico F: 1705.6561
p-valor: 0.0000e+00
Conclusión: Se rechaza H0. Existen diferencias estadísticamente significativas en el número medio de clientes entre las tiendas.


4- Finalmente, identifiquen la tienda con mayor promedio y la tienda con menor promedio de clientes y realicen una prueba de hipótesis para determinar si la diferencia entre ellas es distinta de cero o no. Verifiquen si las tiendas identificadas corresponden a las tiendas con mayores y menores efectos.

In [6]:
# 1. Identificar las tiendas de mayor y menor promedio
promedios = df_simulacion.groupby('Tienda')['Clientes'].mean()

tienda_max = promedios.idxmax()
tienda_min = promedios.idxmin()

datos_max = df_simulacion[df_simulacion['Tienda'] == tienda_max]['Clientes']
datos_min = df_simulacion[df_simulacion['Tienda'] == tienda_min]['Clientes']

# 2. Prueba t para diferencia de medias (exactamente como en el notebook de clase)
t_stat, p_val = stats.ttest_ind(datos_max, datos_min)

print("=== PRUEBA DE HIPÓTESIS (Diferencia de Medias) ===")
print(f"Tienda con mayor promedio: {tienda_max} ({promedios[tienda_max]:.2f})")
print(f"Tienda con menor promedio: {tienda_min} ({promedios[tienda_min]:.2f})")
print(f"Estadístico t: {t_stat:.4f}")
print(f"p-valor: {p_val:.4e}")

if p_val < 0.05:
    print("Conclusión: Se rechaza H0. La diferencia de clientes entre ambas tiendas es estadísticamente distinta de cero.")
else:
    print("Conclusión: No se rechaza H0. No hay evidencia de diferencia entre ambas tiendas.")

=== PRUEBA DE HIPÓTESIS (Diferencia de Medias) ===
Tienda con mayor promedio: Santa Ana (10716.98)
Tienda con menor promedio: Palermo (6722.19)
Estadístico t: 78.7505
p-valor: 0.0000e+00
Conclusión: Se rechaza H0. La diferencia de clientes entre ambas tiendas es estadísticamente distinta de cero.
